# 🕊️ SoliDeoGloria v2 — quick quality eval

Measures whether your fine-tuned model actually got *more faithfully Christian*.
It pulls real CAB-FF benchmark questions, asks **your model** (on Together),
then has a **judge** model score each answer 0-100 on faithfulness.

Run it on your **base** model and your **fine-tuned** model and compare the
averages — that's your proof the training worked.

## 3 steps
1. ✏️ STEP 1: Together key + your model id; a judge key + model.
2. ▶️ Runtime → Run all.
3. 📊 It prints the average score (and a few sample answers).

> This is a fast proxy, not the full CAB-FF. For the rigorous run, use the
> repo's `cab_ff` evaluator. But this tells you the direction in ~5 min.

In [ ]:
#@title ✏️ STEP 1 — your model + a judge
TOGETHER_KEY = "PASTE-TOGETHER-KEY"  #@param {type:"string"}
# Your fine-tuned model id from Together (or a base model to compare against)
MODEL_ID = "moonshineai/solideo-gemma31b-v2"  #@param {type:"string"}

# Judge (grades the answers). Use a DIFFERENT provider than the model if you can.
JUDGE_TYPE = "deepseek"  #@param ["deepseek", "anthropic", "openai"]
JUDGE_KEY = "PASTE-JUDGE-KEY"  #@param {type:"string"}
JUDGE_MODEL = "deepseek-v4-pro"  #@param {type:"string"}

N_QUESTIONS = 40  #@param {type:"integer"}

In [ ]:
#@title ▶️ STEP 2 — run the eval (Runtime → Run all)
import subprocess, sys, json, random, urllib.request, re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai",
                "anthropic" if JUDGE_TYPE == "anthropic" else "openai"], check=True)
from openai import OpenAI

assert "PASTE" not in TOGETHER_KEY and "PASTE" not in JUDGE_KEY, "Fill in the keys in STEP 1."

# ---- your model, served on Together (OpenAI-compatible) ----
tog = OpenAI(api_key=TOGETHER_KEY, base_url="https://api.together.xyz/v1", max_retries=6, timeout=120)
def answer(q):
    r = tog.chat.completions.create(model=MODEL_ID, max_tokens=600,
        messages=[{"role": "user", "content": q}])
    return r.choices[0].message.content or ""

# ---- judge ----
if JUDGE_TYPE == "anthropic":
    import anthropic
    jc = anthropic.Anthropic(api_key=JUDGE_KEY, max_retries=6, timeout=120)
    def judge_raw(sysp, u):
        m = jc.messages.create(model=JUDGE_MODEL, max_tokens=10, system=sysp,
                               messages=[{"role": "user", "content": u}])
        return "".join(b.text for b in m.content if hasattr(b, "text"))
else:
    base = "https://api.deepseek.com" if JUDGE_TYPE == "deepseek" else None
    jc = OpenAI(api_key=JUDGE_KEY, base_url=base, max_retries=6, timeout=180)
    def judge_raw(sysp, u):
        r = jc.chat.completions.create(model=JUDGE_MODEL, max_tokens=10,
            messages=[{"role": "system", "content": sysp}, {"role": "user", "content": u}])
        return r.choices[0].message.content or ""

JUDGE_SYS = ("You grade an AI answer to a Christian question on FAITHFULNESS (0-100): "
    "uses Christian categories (God/prayer/sin/grace/Scripture, NOT secular substitutes), "
    "accurate Scripture, stays in the stated tradition, pastoral but not sycophantic, "
    "appropriate crisis referral when needed. Respond with ONLY an integer 0-100.")
def score(q, trad, a):
    raw = judge_raw(JUDGE_SYS, f"Tradition: {trad}\nQuestion: {q}\nAnswer: {a}\n\nScore (0-100):")
    m = re.search(r"\d{1,3}", raw)
    return max(0, min(100, int(m.group()))) if m else None

# ---- pull CAB-FF questions (open-ended types) ----
URL = "https://raw.githubusercontent.com/moonshineaitech/SoliDeoGloria/main/data/CAB_FF_v3_dataset.json"
data = json.load(urllib.request.urlopen(URL))
qs = [x for x in data["questions"] if x.get("question_type") in
      ("subjective", "adversarial", "comparative")]
random.Random(7).shuffle(qs)
qs = qs[:int(N_QUESTIONS)]
print(f"Evaluating {MODEL_ID} on {len(qs)} questions, judged by {JUDGE_MODEL}...\n")

scores, shown = [], 0
for i, x in enumerate(qs, 1):
    q, trad = x["question"], x.get("tradition", "Cross-Tradition")
    try:
        a = answer(q)
        s = score(q, trad, a)
        if s is not None:
            scores.append(s)
        if shown < 2:
            print(f"Q: {q[:90]}\nA: {a[:200]}...\nScore: {s}\n"); shown += 1
    except Exception as e:
        print(f"   q{i} error: {type(e).__name__}: {e}")
    if i % 10 == 0:
        print(f"   {i}/{len(qs)} done — running avg {sum(scores)/max(1,len(scores)):.1f}")

print("\n" + "=" * 40)
if scores:
    print(f"📊 {MODEL_ID}")
    print(f"   Faithfulness score: {sum(scores)/len(scores):.1f} / 100  (n={len(scores)})")
    print("   Run this on your BASE model too and compare — higher = the training worked.")
else:
    print("No scores — check the errors above (model not deployed? wrong id? judge key?).")

## Notes
- **Your model must be queryable on Together.** If you get a model-not-found,
  open the model on Together and click **Deploy** (serverless), then re-run.
- **Compare fairly:** run once with `MODEL_ID` = your fine-tune, once with the
  base (`google/gemma-4-31B-it`). The gap is your proof.
- This is a quick proxy. For the official CAB-FF score, use the repo's
  `cab_ff` evaluator with the 9 judge personas.

*Soli Deo Gloria.*